In [1]:
# Model Librari
import pandas as pd
import re
import string
import requests
import ast
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
from pathlib import Path
import pandas as pd

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
source = None
target_names = [
    'hasil_preprosesing_aveta_tr.csv',
    'hasil_preprosesing_aveta.csv'
]

for candidate in candidates:
    for name in target_names:
        direct = candidate / 'data' / 'prosses' / name
        nested = candidate / 'analisis' / 'data' / 'prosses' / name
        if direct.exists():
            source = direct
            break
        if nested.exists():
            source = nested
            break
    if source is not None:
        break

if source is None:
    raise FileNotFoundError(
        'File preprocessing tidak ditemukan. Cek folder data/prosses atau analisis/data/prosses.'
    )

data = pd.read_csv(source)
data = data.loc[:, ~data.columns.str.contains('^Unnamed')]
ACTIVE_RAW_DATASET = str(source)
print(f"Raw dataset aktif: {source}")
print(f"Total data: {len(data)}")
data.head()


Raw dataset aktif: D:\PROYEK TUGAS AKHIR\APP TA\analisis\data\prosses\hasil_preprosesing_aveta_tr.csv
Total data: 2179


,rating,ulasan,label,cleaning,tokenize,normalisasi,lematisasi,stopword,stemming,compare,jumlah_kata
0,5/5,Hotel yang sangat strategis di Jalan Malioboro...,Negatif,hotel yang sangat strategis di jalan malioboro...,"['hotel', 'yang', 'sangat', 'strategis', 'di',...","['hotel', 'yang', 'sangat', 'mudah ditemukan',...","['hotel', 'yang', 'sangat', 'mudah', 'temukan'...","['hotel', 'mudah', 'temukan', 'jalan', 'maliob...","['hotel', 'mudah', 'temu', 'jalan', 'malioboro...","[('hotel', 'hotel'), ('mudah', 'mudah'), ('tem...",63
1,5/5,Lokasi teratas. Di sekitar lokasi hotel banyak...,Positif,lokasi teratas di sekitar lokasi hotel banyak ...,"['lokasi', 'teratas', 'di', 'sekitar', 'lokasi...","['lokasi', 'teratas', 'di', 'sekitar', 'lokasi...","['lokasi', 'teratas', 'di', 'sekitar', 'lokasi...","['lokasi', 'teratas', 'lokasi', 'hotel', 'bany...","['lokasi', 'atas', 'lokasi', 'hotel', 'banyak'...","[('lokasi', 'lokasi'), ('teratas', 'atas'), ('...",28
2,2/5,Saya tidak cukup merekomendasikannya. Memesan ...,Negatif,saya tidak cukup merekomendasikanya memesan ka...,"['saya', 'tidak', 'cukup', 'merekomendasikanya...","['saya', 'tidak', 'cukup', 'merekomendasikanya...","['saya', 'tidak', 'cukup', 'merekomendasikanya...","['tidak', 'merekomendasikanya', 'memesan', 'ka...","['tidak', 'merekomendasikanya', 'mes', 'kamar'...","[('tidak', 'tidak'), ('merekomendasikanya', 'm...",60
3,5/5,Menginap 2 hari 2 malam bersama istri dan anak...,Positif,menginap hari malam bersama istri dan anak pua...,"['menginap', 'hari', 'malam', 'bersama', 'istr...","['menginap', 'hari', 'malam', 'bersama', 'istr...","['menginap', 'hari', 'malam', 'bersama', 'istr...","['menginap', 'malam', 'istri', 'anak', 'puas',...","['inap', 'malam', 'istri', 'anak', 'puas', 'ba...","[('menginap', 'inap'), ('malam', 'malam'), ('i...",36
4,2/5,Saya memberikan ini 2 bintang sebelum saya mem...,Negatif,saya memberikan ini bintang sebelum saya memas...,"['saya', 'memberikan', 'ini', 'bintang', 'sebe...","['saya', 'memberikan', 'ini', 'bintang', 'sebe...","['saya', 'memberikan', 'ini', 'bintang', 'sebe...","['bintang', 'memasuki', 'ruangan', 'ruangan', ...","['bintang', 'pasuk', 'ruang', 'ruang', 'jam', ...","[('bintang', 'bintang'), ('memasuki', 'pasuk')...",62


CASEFOLDING

In [6]:
def case_folding(text):
    if isinstance(text, str):
        return text.lower()
    return ""

data['casefolding'] = data['ulasan'].apply(case_folding)
data[['ulasan', 'casefolding']].head()


,ulasan,casefolding
0,Hotel yang sangat strategis di Jalan Malioboro...,hotel yang sangat strategis di jalan malioboro...
1,Lokasi teratas. Di sekitar lokasi hotel banyak...,lokasi teratas. di sekitar lokasi hotel banyak...
2,Saya tidak cukup merekomendasikannya. Memesan ...,saya tidak cukup merekomendasikannya. memesan ...
3,Menginap 2 hari 2 malam bersama istri dan anak...,menginap 2 hari 2 malam bersama istri dan anak...
4,Saya memberikan ini 2 bintang sebelum saya mem...,saya memberikan ini 2 bintang sebelum saya mem...


In [17]:
i = 22
print("ulasan :", data['ulasan'].iloc[i])
print("casefolding :", data['casefolding'].iloc[i])
print("cleaning :", data['cleaning'].iloc[i])
print("tokenize :", data['tokenize'].iloc[i])
print("normalisasi :", data['normalisasi'].iloc[i])
print("stopword :", data['stopword'].iloc[i])
print("stemming :", data['stemming'].iloc[i])
print("label :", data['label'].iloc[i])


ulasan : Listrik padam sampai 20 menit, tidak ada penjelasan dari pihak hotel, pelayanan kurang memuaskan.
casefolding : listrik padam sampai 20 menit, tidak ada penjelasan dari pihak hotel, pelayanan kurang memuaskan.
cleaning : listrik padam sampai menit tidak ada penjelasan dari pihak hotel pelayanan kurang memuaskan
tokenize : ['listrik', 'padam', 'sampai', 'menit', 'tidak', 'ada', 'penjelasan', 'dari', 'pihak', 'hotel', 'pelayanan', 'kurang', 'memuaskan']
normalisasi : ['listrik', 'padam', 'sampai', 'menit', 'tidak', 'ada', 'penjelasan', 'dari', 'pihak', 'hotel', 'pelayanan', 'kurang', 'sangat puas']
stopword : ['listrik', 'padam', 'menit', 'tidak', 'penjelasan', 'hotel', 'pelayanan', 'kurang', 'puas']
stemming : ['listrik', 'padam', 'menit', 'tidak', 'jelas', 'hotel', 'layan', 'kurang', 'puas']
label : Negatif
